# Figure S1F — Binary vs Confidence F1 Scores (300-Patient Test Set)

Grouped bar chart comparing F1 scores (with 95% stratified bootstrap CIs) for probabilistic vs binary LLM pipelines across 6 toxicities, with paired bootstrap p-values.

**Data sources** (same denominator as Fig 1B):
- Locked test split `test_mrns.csv` (n=298)
- `llama_maverick_test_298_results.csv` — probabilistic predictions
- `combined_298_patients_binary_results.csv` — binary predictions (valid 281 + new 17)
- `final_gold_standard_1k.csv` — patient-level gold standard
- `additional_negative_mrns.txt` — 2 RAG-filtered patients, padded as all-negative (no LLM call), bringing the set to 300

Outputs are written to `figure 1/results/supp/`.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

%matplotlib inline

# ---- Hard-fail if Arial isn't actually resolved (no silent fallback) ----
import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")

In [ ]:
ROOT = Path("..").resolve()
FIGURES = ROOT.parent.parent
DATA = FIGURES / "figures_data" / "figure 1" / "data"
RESULTS = ROOT / "results"
(RESULTS / "supp").mkdir(parents=True, exist_ok=True)

PROB_FILE = DATA / "llama_maverick_test_298_results.csv"
BIN_FILE  = DATA / "combined_298_patients_binary_results.csv"
GOLD_FILE = DATA / "final_gold_standard_1k.csv"
TEST_MRN_FILE = DATA / "test_mrns.csv"
ADDITIONAL_NEG_MRN_FILE = DATA / "additional_negative_mrns.txt"

OUT_PATH = RESULTS / "supp" / "Binary_vs_Prob_F1_S1F.pdf"
CSV_OUT  = RESULTS / "supp" / "Binary_vs_Prob_F1_S1F_results.csv"

for p in [PROB_FILE, BIN_FILE, GOLD_FILE, TEST_MRN_FILE, ADDITIONAL_NEG_MRN_FILE]:
    assert p.exists(), f"Missing: {p}"
print("All input files found.")
print(f"Data: {DATA}")
print(f"Prob: {PROB_FILE.name}")
print(f"Bin:  {BIN_FILE.name}")

In [ ]:
# ===========================================================================
# Rebuild from scratch: identify the 17 MRNs needing a retrieval + binary run
# ===========================================================================
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
FIGURES = ROOT.parent.parent
DATA = FIGURES / "figures_data" / "figure 1" / "data"

def _norm_mrn(s):
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(8)

def _read_csv(path):
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        for enc in ("latin-1", "cp1252"):
            try:
                return pd.read_csv(path, encoding=enc, low_memory=False)
            except UnicodeDecodeError:
                continue
        raise

def _mrn_set(path, col=None):
    df = _read_csv(path)
    if col is None:
        col = next(c for c in df.columns if c.lower() in {"mrn", "mrn_str"})
    return set(_norm_mrn(df[col]))

# --- inputs ---
m_test  = _mrn_set(DATA / "test_mrns.csv")                              # locked split
m_train = _mrn_set(DATA / "train_mrns.csv")
m_1k    = _mrn_set(DATA / "llama_maverick_1k_results.csv", "mrn")       # 998, prob arm
m_gold  = _mrn_set(DATA / "final_gold_standard_1k.csv")
m150    = _mrn_set(DATA / "batched_rag_output_150_patients.csv", "mrn") # stale draw,
m148    = _mrn_set(DATA / "batched_rag_output_148_patients.csv", "mrn") #   retrieval done
m_have  = m150 | m148

print(f"locked test split : {len(m_test)}")
print(f"retrieval done    : {len(m_have)}  (stale 298 draw)")
print(f"overlap           : {len(m_test & m_have)}\n")

# --- the 17 ---
m_need = sorted(m_test - m_have)

assert len(m_need) == 17,           f"Expected 17, got {len(m_need)}"
assert not (set(m_need) & m_have),  "Overlap with completed retrieval"
assert set(m_need) <= m_test,       "Not all in locked test split"
assert set(m_need) <= m_1k,         "Missing probabilistic results"
assert set(m_need) <= m_gold,       "Missing gold standard labels"
assert not (set(m_need) & m_train), "Contamination: MRN in train split"
print("All provenance checks passed.\n")

out = DATA / "rerun_17_mrns.csv"
pd.DataFrame({"MRN": m_need}).to_csv(out, index=False)
print(f"Wrote {out.name} ({len(m_need)} patients)")

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
FIGURES = ROOT.parent.parent
DATA = FIGURES / "figures_data" / "figure 1" / "data"

PROB_FILE  = DATA / "llama_maverick_1k_results.csv"
BIN_FILE   = DATA / "combined_298_patients_binary_results.csv"
GOLD_FILE  = DATA / "final_gold_standard_1k.csv"
TEST_FILE  = DATA / "test_mrns.csv"
TRAIN_FILE = DATA / "train_mrns.csv"
ADDL_NEG   = DATA / "additional_negative_mrns.txt"

# Retrieval outputs from the stale draw -- cross-check only, not authoritative
RAG_FILES = [DATA / "batched_rag_output_150_patients.csv",
             DATA / "batched_rag_output_148_patients.csv"]

TOXICITIES = ["liver toxicity", "hypothyroidism", "pneumonitis",
              "colitis", "adrenal insufficiency", "hyperthyroidism"]


# ---------------------------------------------------------------------------
# Helpers -- identical normalization to S1F / Fig1B (zfill(8), strip .0)
# ---------------------------------------------------------------------------
def _norm_mrn(s):
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(8)


def _read_csv(path):
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        for enc in ("latin-1", "cp1252"):
            try:
                return pd.read_csv(path, encoding=enc, low_memory=False)
            except UnicodeDecodeError:
                continue
        raise


def _mrn_col(df):
    return next(c for c in df.columns if c.lower() in {"mrn", "mrn_str"})


def _mrn_set(path):
    df = _read_csv(path)
    return set(_norm_mrn(df[_mrn_col(df)]))


# ---------------------------------------------------------------------------
# 1. Load the sets
# ---------------------------------------------------------------------------
m_test  = _mrn_set(TEST_FILE)
m_train = _mrn_set(TRAIN_FILE)
m_prob  = _mrn_set(PROB_FILE)
m_gold  = _mrn_set(GOLD_FILE)

bin_df = _read_csv(BIN_FILE)
bin_df["_mrn"] = _norm_mrn(bin_df[_mrn_col(bin_df)])
m_bin = set(bin_df["_mrn"])

addl_neg = [ln.strip().zfill(8) for ln in ADDL_NEG.read_text().splitlines() if ln.strip()]

print("=" * 72)
print("SET SIZES")
print("=" * 72)
print(f"  locked test split (test_mrns.csv)  : {len(m_test):>5}")
print(f"  train split                        : {len(m_train):>5}")
print(f"  binary results                     : {len(m_bin):>5}  "
      f"({len(bin_df):,} rows)")
print(f"  probabilistic results              : {len(m_prob):>5}")
print(f"  gold standard                      : {len(m_gold):>5}")
print(f"  RAG-filtered padding patients      : {len(addl_neg):>5}")
print(f"  Fig1B denominator                  : {len(m_test) + len(addl_neg):>5}")

assert len(addl_neg) == 2, f"Expected 2 padding MRNs, got {len(addl_neg)}"
assert not (set(addl_neg) & m_test), "Padding MRN already in locked split -- padding would double-count"


# ---------------------------------------------------------------------------
# 2. Partition
# ---------------------------------------------------------------------------
overlap    = sorted(m_test & m_bin)     # keep, binary already exists
need_rerun = sorted(m_test - m_bin)     # rerun through RAG + binary
drop       = sorted(m_bin - m_test)     # discard: training-split contamination

print()
print("=" * 72)
print("PARTITION")
print("=" * 72)
print(f"  in locked split AND binary   : {len(overlap):>5}   <- keep as-is")
print(f"  in locked split, NO binary   : {len(need_rerun):>5}   <- RERUN")
print(f"  in binary, NOT locked split  : {len(drop):>5}   <- DROP")
print(f"  {len(overlap)} + {len(need_rerun)} = {len(overlap) + len(need_rerun)} "
      f"(+{len(addl_neg)} padded) = {len(overlap) + len(need_rerun) + len(addl_neg)}")

assert len(overlap) + len(need_rerun) == len(m_test), "Partition does not reconstruct the locked split"


# ---------------------------------------------------------------------------
# 3. Verify binary coverage is COMPLETE for the 281
# ---------------------------------------------------------------------------
print()
print("=" * 72)
print(f"BINARY COVERAGE CHECK ON THE {len(overlap)} OVERLAP")
print("=" * 72)

missing_cols = [t for t in TOXICITIES if t not in bin_df.columns]
if missing_cols:
    print(f"  !! Toxicity columns absent from binary file: {missing_cols}")
    print(f"     Columns present: {sorted(bin_df.columns)}")
    raise SystemExit("Resolve column naming before proceeding.")

sub = bin_df[bin_df["_mrn"].isin(overlap)]

# Collapse note-level -> patient-level if the file is not already one row per MRN
rows_per_mrn = sub.groupby("_mrn").size()
if rows_per_mrn.max() > 1:
    print(f"  Note-level file detected ({rows_per_mrn.max()} rows max per MRN) "
          f"-- collapsing with groupby().max()")
    sub_pat = sub.groupby("_mrn")[TOXICITIES].max()
else:
    sub_pat = sub.set_index("_mrn")[TOXICITIES]

print(f"  patients after collapse : {len(sub_pat)}")

coverage_ok = True
for tox in TOXICITIES:
    n_null = int(sub_pat[tox].isna().sum())
    vals = sub_pat[tox].dropna().unique()
    non_binary = sorted(v for v in vals if v not in (0, 1, 0.0, 1.0))
    flag = ""
    if n_null:
        flag += f"  !! {n_null} NULL"
        coverage_ok = False
    if non_binary:
        flag += f"  !! non-0/1 values: {non_binary[:5]}"
    print(f"    {tox:<24} n={len(sub_pat[tox].dropna()):>4}  "
          f"positives={int(sub_pat[tox].fillna(0).sum()):>3}{flag}")

assert set(sub_pat.index) == set(overlap), "Binary file does not cover every overlap MRN"
assert coverage_ok, "Binary results incomplete on the overlap -- these MRNs also need a rerun"
print(f"\n  PASS: complete binary coverage on all {len(overlap)} overlap patients.")


# ---------------------------------------------------------------------------
# 4. Provenance checks on the 17
# ---------------------------------------------------------------------------
print()
print("=" * 72)
print(f"PROVENANCE CHECKS ON THE {len(need_rerun)} RERUN MRNs")
print("=" * 72)

checks = [
    ("all in locked test split",        set(need_rerun) <= m_test),
    ("none in train split",             not (set(need_rerun) & m_train)),
    ("none already in binary results",  not (set(need_rerun) & m_bin)),
    ("all have probabilistic results",  set(need_rerun) <= m_prob),
    ("all have gold standard labels",   set(need_rerun) <= m_gold),
    ("none are padding MRNs",           not (set(need_rerun) & set(addl_neg))),
]
for label, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
assert all(ok for _, ok in checks), "Provenance check failed -- do not queue the rerun"

# Cross-check against retrieval outputs (informational)
m_rag = set()
for f in RAG_FILES:
    if f.exists():
        m_rag |= _mrn_set(f)
if m_rag:
    already_retrieved = sorted(set(need_rerun) & m_rag)
    print(f"\n  Retrieval cross-check ({len(m_rag)} MRNs in batched_rag_output_*):")
    if already_retrieved:
        print(f"    {len(already_retrieved)} of the rerun set already have retrieval done "
              f"-- binary classification only, no RAG needed.")
    else:
        print(f"    0 of the 17 have retrieval -- all 17 need full RAG + binary.")
    print(f"    binary set == retrieval set: {m_bin == m_rag}")


# ---------------------------------------------------------------------------
# 5. Write outputs
# ---------------------------------------------------------------------------
print()
print("=" * 72)
print("OUTPUTS")
print("=" * 72)

out_rerun   = DATA / "s1f_rerun_17_mrns.csv"
out_overlap = DATA / "s1f_overlap_281_mrns.csv"
out_drop    = DATA / "s1f_drop_17_mrns.csv"

pd.DataFrame({"MRN": need_rerun}).to_csv(out_rerun, index=False)
pd.DataFrame({"MRN": overlap}).to_csv(out_overlap, index=False)
pd.DataFrame({"MRN": drop}).to_csv(out_drop, index=False)

print(f"  {out_rerun.name:<28} {len(need_rerun):>4} patients  <- hand to the RAG rerun")
print(f"  {out_overlap.name:<28} {len(overlap):>4} patients")
print(f"  {out_drop.name:<28} {len(drop):>4} patients  (contaminated, discard)")

print(f"\n  After the rerun, S1F test set = {len(overlap)} + {len(need_rerun)} "
      f"+ {len(addl_neg)} padded = {len(overlap) + len(need_rerun) + len(addl_neg)}, "
      f"identical to Fig1B.")

In [ ]:
# ---------------------------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------------------------
N_BOOT = 2000
SEED   = 42

THRESHOLDS = {
    "pneumonitis":           0.710,
    "adrenal_insufficiency": 0.810,
    "liver_toxicity":        0.010,
    "colitis":               0.710,
    "hyperthyroidism":       0.810,
    "hypothyroidism":        0.510,
}

# Display order matched across all panels
TOXICITIES = ["liver_toxicity", "hypothyroidism", "pneumonitis",
              "colitis", "adrenal_insufficiency", "hyperthyroidism"]
assert set(TOXICITIES) == set(THRESHOLDS), "TOXICITIES out of sync with THRESHOLDS"

DISPLAY_NAMES = {
    "pneumonitis":           "Pneumonitis",
    "adrenal_insufficiency": "Adrenal\ninsufficiency",
    "liver_toxicity":        "Liver\ntoxicity",
    "colitis":               "Colitis",
    "hyperthyroidism":       "Hyper-\nthyroidism",
    "hypothyroidism":        "Hypo-\nthyroidism",
}

PROB_COLOR = "#4878CF"
BIN_COLOR  = "#D65F5F"

In [ ]:
def _norm_mrn(s):
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(8)

def _read_csv(path):
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        for enc in ("latin-1", "cp1252"):
            try:
                return pd.read_csv(path, encoding=enc, low_memory=False)
            except UnicodeDecodeError:
                continue
        raise


In [ ]:
def f1_at_threshold(y_true, y_score, thr):
    y_pred = (y_score >= (thr - 1e-10)).astype(int)
    return float(f1_score(y_true, y_pred, zero_division=0))

def format_pval(p):
    if p < 0.001:
        return "P < 0.001"
    elif p < 0.01:
        return f"P = {p:.3f}"
    elif p < 0.05:
        return f"P = {p:.2f}"
    else:
        return f"P = {p:.2f}"


In [ ]:
# ---------------------------------------------------------------------------
# Assemble patient-level scores, then pad the 2 RAG-filtered patients 
# ---------------------------------------------------------------------------
COL_RENAME = {
    "liver toxicity": "liver_toxicity",
    "adrenal insufficiency": "adrenal_insufficiency",
}

def _patient_max(path):
    df = _read_csv(path)
    df = df.rename(columns=COL_RENAME)
    df["mrn"] = _norm_mrn(df["mrn"])
    missing = [t for t in TOXICITIES if t not in df.columns]
    if missing:
        raise KeyError(f"{path.name} missing columns: {missing}")
    return df.groupby("mrn")[TOXICITIES].max()

prob_scores = _patient_max(PROB_FILE)
bin_scores  = _patient_max(BIN_FILE)

gold = _read_csv(GOLD_FILE).rename(columns=COL_RENAME)
gold["mrn"] = _norm_mrn(gold["mrn"])
gold = gold.drop_duplicates("mrn").set_index("mrn")
prob_truth = gold.reindex(columns=TOXICITIES).apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

test_df = _read_csv(TEST_MRN_FILE)
test_col = next(c for c in test_df.columns if c.lower() in {"mrn", "mrn_str"})
test_ids = list(dict.fromkeys(_norm_mrn(test_df[test_col]).tolist()))

assert set(test_ids) <= set(prob_scores.index), "Probability file missing locked-test patients"
assert set(test_ids) <= set(bin_scores.index),  "Binary file missing locked-test patients"
assert set(test_ids) <= set(prob_truth.index),  "Gold standard missing locked-test patients"

prob_scores = prob_scores.loc[test_ids]
bin_scores  = bin_scores.loc[test_ids]
prob_truth  = prob_truth.loc[test_ids]

# Same padding as Fig 1B: these patients never reached the LLM, so every
# toxicity is a negative prediction (0.0) and a negative label (0).
ADDITIONAL_NEG_MRNS = [
    line.strip().zfill(8)
    for line in ADDITIONAL_NEG_MRN_FILE.read_text().splitlines() if line.strip()
]
assert len(ADDITIONAL_NEG_MRNS) == 2, f"Expected 2 padding patients, got {len(ADDITIONAL_NEG_MRNS)}"
assert not (set(ADDITIONAL_NEG_MRNS) & set(test_ids)), "Padding patient already in locked split"

n_added = 0
for mrn in ADDITIONAL_NEG_MRNS:
    if mrn in prob_scores.index:
        continue
    zero_score = pd.DataFrame([[0.0] * len(TOXICITIES)], columns=TOXICITIES, index=[mrn])
    zero_truth = zero_score.astype(int)
    prob_scores = pd.concat([prob_scores, zero_score])
    bin_scores  = pd.concat([bin_scores, zero_score])
    prob_truth  = pd.concat([prob_truth, zero_truth])
    n_added += 1

assert prob_scores.index.equals(bin_scores.index)
assert prob_scores.index.equals(prob_truth.index)
print(f"Locked test: {len(test_ids)}")
print(f"Padded RAG-filtered all-negative patients: {n_added}")
print(f"Test set size after additions: {len(prob_scores)}")
assert len(prob_scores) == 300, f"Expected 300, got {len(prob_scores)}"


In [ ]:
# ---------------------------------------------------------------------------
# Compute metrics + paired bootstrap p-values
# ---------------------------------------------------------------------------
results_prob = []
results_bin  = []
p_values     = []
stat_rows    = []

for tox in TOXICITIES:
    thr = THRESHOLDS[tox]
    y_true  = prob_truth[tox].values.astype(int)
    s_prob  = prob_scores[tox].values.astype(float)
    s_bin   = bin_scores[tox].values.astype(float)

    f1_prob_point = f1_at_threshold(y_true, s_prob, thr)
    f1_bin_point  = f1_at_threshold(y_true, s_bin, thr)

    # Paired bootstrap: same indices for both
    rng = np.random.default_rng(SEED)
    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]

    boot_prob = np.empty(N_BOOT)
    boot_bin  = np.empty(N_BOOT)
    for b in range(N_BOOT):
        s_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True) if len(pos_idx) > 0 else pos_idx
        s_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        idx = np.concatenate([s_pos, s_neg])
        boot_prob[b] = f1_at_threshold(y_true[idx], s_prob[idx], thr)
        boot_bin[b]  = f1_at_threshold(y_true[idx], s_bin[idx], thr)

    prob_lo, prob_hi = np.percentile(boot_prob, 2.5), np.percentile(boot_prob, 97.5)
    bin_lo, bin_hi   = np.percentile(boot_bin, 2.5), np.percentile(boot_bin, 97.5)

    # Two-sided paired bootstrap p-value
    delta = boot_prob - boot_bin
    p_val = 2 * min(np.mean(delta >= 0), np.mean(delta <= 0))
    p_val = min(p_val, 1.0)

    results_prob.append({"tox": tox, "F1": f1_prob_point, "lo": prob_lo, "hi": prob_hi})
    results_bin.append({"tox": tox, "F1": f1_bin_point, "lo": bin_lo, "hi": bin_hi})
    p_values.append(p_val)

    print(f"  {tox:<25} prob F1={f1_prob_point:.3f} [{prob_lo:.3f}-{prob_hi:.3f}]  "
          f"bin F1={f1_bin_point:.3f} [{bin_lo:.3f}-{bin_hi:.3f}]  "
          f"p={p_val:.4f}  {format_pval(p_val)}")

    stat_rows.append({
        "panel": "S1F",
        "ae_type": tox,
        "n": int(len(y_true)),
        "n_positive": int(y_true.sum()),
        "threshold": thr,
        "f1_confidence": f1_prob_point,
        "f1_ci_lower_confidence": float(prob_lo),
        "f1_ci_upper_confidence": float(prob_hi),
        "f1_binary": f1_bin_point,
        "f1_ci_lower_binary": float(bin_lo),
        "f1_ci_upper_binary": float(bin_hi),
        "p_paired": p_val,
    })

CSV_OUT.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(stat_rows).to_csv(CSV_OUT, index=False)
print(f"Saved CSV: {CSV_OUT.name}")


In [ ]:
# ---------------------------------------------------------------------------
# Figure with p-value brackets
# ---------------------------------------------------------------------------
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(3.6, 2.0))
x = np.arange(len(TOXICITIES))
w = 0.36

prob_f1  = np.array([r["F1"] for r in results_prob])
prob_lo  = np.array([r["lo"] for r in results_prob])
prob_hi  = np.array([r["hi"] for r in results_prob])
prob_yerr = np.array([np.maximum(0.0, prob_f1 - prob_lo), np.maximum(0.0, prob_hi - prob_f1)])

bin_f1  = np.array([r["F1"] for r in results_bin])
bin_lo  = np.array([r["lo"] for r in results_bin])
bin_hi  = np.array([r["hi"] for r in results_bin])
bin_yerr = np.array([np.maximum(0.0, bin_f1 - bin_lo), np.maximum(0.0, bin_hi - bin_f1)])

bars_prob = ax.bar(x - w/2, prob_f1, width=w, yerr=prob_yerr, capsize=3,
       color=PROB_COLOR, edgecolor="white", linewidth=0.5,
       error_kw={"linewidth": 0.8}, label="Confidence")
bars_bin = ax.bar(x + w/2, bin_f1, width=w, yerr=bin_yerr, capsize=3,
       color=BIN_COLOR, edgecolor="white", linewidth=0.5,
       error_kw={"linewidth": 0.8}, label="Binary")

# --- P-value brackets ---
bracket_color = "black"
bracket_lw = 0.8

for i in range(len(TOXICITIES)):
    top_prob = prob_f1[i] + prob_yerr[1, i]
    top_bin  = bin_f1[i] + bin_yerr[1, i]
    bracket_y = max(top_prob, top_bin) + 0.03
    bracket_top = bracket_y + 0.02

    x_left  = x[i] - w/2
    x_right = x[i] + w/2

    ax.plot([x_left, x_left, x_right, x_right],
            [bracket_y, bracket_top, bracket_top, bracket_y],
            color=bracket_color, linewidth=bracket_lw, clip_on=False)

    ax.text((x_left + x_right) / 2, bracket_top + 0.01,
            format_pval(p_values[i]),
            ha="center", va="bottom", fontsize=5, fontfamily="Arial",
            fontstyle="italic")

ax.set_ylabel("F1 Score")
ax.set_xticks(x)
ax.set_xticklabels([DISPLAY_NAMES[t] for t in TOXICITIES])
ax.set_ylim(0, 1.25)
ax.set_yticks(np.arange(0, 1.1, 0.2))
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("bottom", "left"):
    ax.spines[spine].set_linewidth(0.6)
ax.tick_params(width=0.6, length=3)
ax.legend(frameon=False, loc="upper right")

plt.tight_layout()
fig.savefig(OUT_PATH, format="pdf", dpi=450)
print(f"Saved: {OUT_PATH}")
plt.show()